# Mizo NER — quickstart

Tag Mizo text with the recognizer, load the corpus, and reproduce the score
reported against human annotation.

```
pip install transformers torch
```

Download `mizo_ner_corpus.jsonl` and `mizo_ner_gold.json` from
https://huggingface.co/datasets/haulai/mizo-ner into this folder.

## Tag a sentence

In [ ]:
from transformers import pipeline

ner = pipeline("token-classification",
               model="haulai/mizo-ner-mizbert",
               aggregation_strategy="simple")

examples = [
    "Pu Lalthanhawla chuan Aizawlah thu a sawi.",
    "Mizo tawng hi Tibeto-Burman \u1e6dawng a ni.",
    "MZU leh Serchhip College chuan an thawh ho.",
]
for text in examples:
    print(text)
    for e in ner(text):
        print("   {:<26} {:<12} {:.3f}".format(
              e["word"], e["entity_group"], e["score"]))
    print()

## Load the corpus

JSON Lines, one sentence per line, character-offset spans.

In [ ]:
import json
from collections import Counter

rows = []
with open("mizo_ner_corpus.jsonl", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

print("{:,} sentences".format(len(rows)))
r = rows[0]
print("text    :", r["text"])
print("entities:", r["entities"])        # [start, end, label]
print("english :", r["english_source"])

c = Counter(e[2] for x in rows for e in x["entities"])
print("\n{:,} entities".format(sum(c.values())))
for k, v in c.most_common():
    print("  {:<14}{:>8,}".format(k, v))

## Reproduce the gold evaluation

300 sentences annotated by two Mizo speakers and adjudicated.

In [ ]:
import json, torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL = "haulai/mizo-ner-mizbert"
gold = json.load(open("mizo_ner_gold.json", encoding="utf-8"))
n_ent = sum(1 for r in gold for t in r["tags"] if t.startswith("B-"))
print("{} sentences, {:,} entities".format(len(gold), n_ent))

dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
mdl = AutoModelForTokenClassification.from_pretrained(MODEL).to(dev).eval()
id2tag = {int(k): v for k, v in mdl.config.id2label.items()}

def predict(tokens):
    enc = tok(tokens, is_split_into_words=True, truncation=True,
              max_length=96, return_tensors="pt").to(dev)
    with torch.no_grad():
        p = torch.argmax(mdl(**enc).logits, dim=2)[0].cpu().numpy()
    out, seen = ["O"] * len(tokens), set()
    for i, w in enumerate(enc.word_ids(0)):
        if w is not None and w not in seen:
            seen.add(w); out[w] = id2tag[int(p[i])]
    return out

def spans(tags):
    out, i = [], 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            lab, j = tags[i][2:], i + 1
            while j < len(tags) and tags[j] == "I-" + lab:
                j += 1
            out.append((i, j, lab)); i = j
        else:
            i += 1
    return out

# spans are matched by overlap: the corpus marks stems, the gold set marks
# whole inflected tokens
tp = fp = fn = 0
for r in gold:
    R, H = spans(r["tags"]), spans(predict(r["tokens"]))
    used = set()
    for hs, he, hl in H:
        hit = None
        for k, (rs, re_, rl) in enumerate(R):
            if k not in used and rl == hl and rs < he and re_ > hs:
                hit = k; break
        if hit is None: fp += 1
        else: used.add(hit); tp += 1
    fn += len(R) - len(used)

p = tp / (tp + fp); rc = tp / (tp + fn); f1 = 2 * p * rc / (p + rc)
print("\nprecision {:.4f}   recall {:.4f}   F1 {:.4f}".format(p, rc, f1))
print("the paper reports F1 0.6414 for this model against the gold set")